# Gullak: BKT input preparation — SYNTHETIC ONLY

Increment 1: validate quiz answer rows and generate 96 synthetic answers (12 learners, four submissions, two questions). No training or real-student accuracy. No app integration; quiz graph pass marks remain the only unlock rule.

Run cells top to bottom on a Colab CPU runtime; no installs, keys, Drive mount or network needed. This is a small data-validation smoke run, not a training job. Later fitting must be separately reviewed.

The nine input columns match the latest R3 contract: event_id, learner_id, topic_id, concept_id, question_id, correct, answered_at, attempt_no, content_version. No question_index column. Content version is SHA-256 of exact topic-content UTF-8 bytes. Use an immutable content archive supplied by R3 to obtain question order and concepts; never reconstruct history from the current answer key.

Each submission shares attempt_no (the topic's quiz_attempts after submission). Within it, questions are ordered by position in pinned content. Missing historical content or ambiguous positions fail closed. The synthetic content below is an engineering fixture with the real money-in / money-out concept IDs, not a reviewed lesson.

Synthetic labels live in notebook and dataset metadata, not extra input columns. Keep future real learner rows out of this publicly shared notebook. Story safe/risky/harmful verdicts are excluded from BKT for now.


In [ ]:
import hashlib
import json
import random
import re
from datetime import datetime, timedelta, timezone
from uuid import UUID

LABEL = "SYNTHETIC — NOT REAL STUDENT DATA"
FIELDS = {"event_id", "learner_id", "topic_id", "concept_id", "question_id",
          "correct", "answered_at", "attempt_no", "content_version"}
# Synthetic teaching fixture, NOT an approved/published money-basics lesson.
SYNTHETIC_CONTENT = {
    "topicId": "money-basics", "locale": "en",
    "title": "SYNTHETIC engineering fixture", "summary": LABEL,
    "variants": [{"id": "synthetic-check", "format": "quiz", "questions": [
        {"id": "money-in", "conceptId": "money-in", "prompt": "Gift arrives: money in or out?",
         "options": [{"id": "in", "text": "In"}, {"id": "out", "text": "Out"}],
         "correctOptionId": "in", "explanation": "A gift adds money."},
        {"id": "money-out", "conceptId": "money-out", "prompt": "Paying a bill: money in or out?",
         "options": [{"id": "in", "text": "In"}, {"id": "out", "text": "Out"}],
         "correctOptionId": "out", "explanation": "A payment takes money away."}
    ]}]
}
# Hash these exact UTF-8 bytes. Never reserialize real content to calculate its hash.
SYNTHETIC_CONTENT_RAW = json.dumps(SYNTHETIC_CONTENT, ensure_ascii=False, separators=(",", ":"))


def synthetic_content_hash():
    return hashlib.sha256(SYNTHETIC_CONTENT_RAW.encode("utf-8")).hexdigest()


def validate_answer_rows(rows, registry):
    """Validate nine-field rows against a trusted, version-pinned question registry.
    Registry keys: (topic_id, content_version, question_id).
    Values: {"concept_id": canonical slug, "position": content question order}.
    Return sorted copies; no guessing, grading, mastery update or mutation.
    """
    if not isinstance(rows, list) or not rows:
        raise ValueError("expected a non-empty list of quiz answers")
    parsed, event_ids, positions, questions, versions = [], set(), set(), set(), {}
    for index, raw in enumerate(rows):
        if not isinstance(raw, dict) or set(raw) != FIELDS:
            raise ValueError(f"row {index}: expected exactly the nine quiz_answers fields")
        row = dict(raw)
        for field in ("event_id", "learner_id"):
            value = row[field]
            if not isinstance(value, str):
                raise ValueError(f"row {index}: invalid {field}")
            try:
                normalized = str(UUID(value))
            except ValueError as error:
                raise ValueError(f"row {index}: invalid {field}") from error
            row[field] = normalized
        for field in ("topic_id", "concept_id", "question_id"):
            if not isinstance(row[field], str) or not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", row[field]):
                raise ValueError(f"row {index}: invalid {field} slug")
        if type(row["correct"]) is not bool:
            raise ValueError(f"row {index}: correct must be boolean")
        if type(row["attempt_no"]) is not int or not 1 <= row["attempt_no"] <= 2147483647:
            raise ValueError(f"row {index}: attempt_no must be a positive Postgres integer")
        if not isinstance(row["content_version"], str) or not re.fullmatch(r"[a-f0-9]{64}", row["content_version"]):
            raise ValueError(f"row {index}: content_version must be lowercase SHA-256 hex")
        if not isinstance(row["answered_at"], str):
            raise ValueError(f"row {index}: timestamp must be text")
        try:
            stamp = datetime.fromisoformat(row["answered_at"].replace("Z", "+00:00"))
        except ValueError as error:
            raise ValueError(f"row {index}: invalid timestamp") from error
        if stamp.tzinfo is None or stamp.utcoffset() is None:
            raise ValueError(f"row {index}: timestamp must include a timezone")
        row["answered_at"] = stamp.astimezone(timezone.utc).isoformat()
        definition = registry.get((row["topic_id"], row["content_version"], row["question_id"]))
        if not isinstance(definition, dict) or definition.get("concept_id") != row["concept_id"]:
            raise ValueError(f"row {index}: unknown pinned question or concept mismatch")
        position = definition.get("position")
        if type(position) is not int or position < 0:
            raise ValueError(f"row {index}: missing trusted question position")
        submission = (row["learner_id"], row["topic_id"], row["attempt_no"])
        if submission in versions and versions[submission] != row["content_version"]:
            raise ValueError(f"row {index}: mixed content versions within a submission")
        versions[submission] = row["content_version"]
        position_key = (*submission, position)
        question_key = (*submission, row["question_id"])
        if row["event_id"] in event_ids or position_key in positions or question_key in questions:
            raise ValueError(f"row {index}: duplicate event, question or ambiguous content order")
        event_ids.add(row["event_id"])
        positions.add(position_key)
        questions.add(question_key)
        parsed.append((row, position))
    parsed.sort(key=lambda pair: (pair[0]["learner_id"], pair[0]["topic_id"], pair[0]["attempt_no"], pair[1]))
    # A whole submission may share a server timestamp. Check between submissions,
    # not within one: content order, rather than timestamp, orders its answers.
    ranges = {}
    for row, _ in parsed:
        key = (row["learner_id"], row["topic_id"], row["attempt_no"])
        stamp = datetime.fromisoformat(row["answered_at"])
        old = ranges.get(key, (stamp, stamp))
        ranges[key] = (min(old[0], stamp), max(old[1], stamp))
    previous = {}
    for (learner, topic, _), (earliest, latest) in sorted(ranges.items()):
        key = (learner, topic)
        if key in previous and earliest < previous[key]:
            raise ValueError("timestamps contradict quiz attempt order")
        previous[key] = latest
    return [row for row, _ in parsed]


def generate_synthetic_responses(seed=42):
    """Fixed small engineering fixture; no model training or accuracy claims."""
    if type(seed) is not int:
        raise ValueError("seed must be an integer")
    rng = random.Random(seed)
    version = synthetic_content_hash()
    quiz = SYNTHETIC_CONTENT["variants"][0]["questions"]
    registry = {("money-basics", version, q["id"]):
                {"concept_id": q["conceptId"], "position": position}
                for position, q in enumerate(quiz)}
    rows = []
    start = datetime(2026, 1, 1, tzinfo=timezone.utc)
    for learner_index in range(12):
        learner = str(UUID(int=rng.getrandbits(128), version=4))
        for attempt in range(1, 5):
            timestamp = (start + timedelta(days=attempt - 1, minutes=learner_index)).isoformat()
            for question in quiz:
                rows.append({
                    "event_id": str(UUID(int=rng.getrandbits(128), version=4)),
                    "learner_id": learner, "topic_id": "money-basics",
                    "concept_id": question["conceptId"], "question_id": question["id"],
                    "correct": rng.random() < 0.25 + attempt * 0.12,
                    "answered_at": timestamp, "attempt_no": attempt, "content_version": version,
                })
    metadata = {"label": LABEL, "synthetic": True, "production_approved": False,
                "seed": seed, "content_version": version,
                "generator": "fixed engineering fixture; not a calibrated learner model"}
    return rows, registry, metadata


In [ ]:
import copy
import unittest

class SyntheticInputTests(unittest.TestCase):
    def setUp(self):
        self.rows, self.registry, self.metadata = generate_synthetic_responses(seed=42)

    def test_valid_rows_match_exact_contract(self):
        result = validate_answer_rows(self.rows, self.registry)
        self.assertEqual(len(result), 96)
        self.assertEqual(set(result[0]), {
            "event_id", "learner_id", "topic_id", "concept_id", "question_id",
            "correct", "answered_at", "attempt_no", "content_version",
        })

    def test_seed_is_reproducible_and_metadata_is_synthetic(self):
        self.assertEqual(generate_synthetic_responses(seed=42), (self.rows, self.registry, self.metadata))
        self.assertNotEqual(generate_synthetic_responses(seed=43)[0], self.rows)
        self.assertTrue(self.metadata["synthetic"])
        self.assertFalse(self.metadata["production_approved"])
        self.assertIn("SYNTHETIC", self.metadata["label"])
        self.assertEqual(self.metadata["seed"], 42)

    def test_each_required_field_is_checked(self):
        for field in self.rows[0]:
            with self.subTest(field=field):
                rows = copy.deepcopy(self.rows)
                del rows[0][field]
                with self.assertRaises(ValueError):
                    validate_answer_rows(rows, self.registry)

    def test_invalid_values_are_rejected(self):
        cases = [
            ("event_id", "not-a-uuid"), ("learner_id", "not-a-uuid"),
            ("topic_id", "Money Basics"), ("concept_id", "Money In"),
            ("question_id", "Question_1"), ("correct", 1), ("correct", "false"),
            ("answered_at", "2026-01-01T00:00:00"), ("answered_at", "not-a-date"),
            ("attempt_no", 0), ("attempt_no", True), ("attempt_no", 1.5),
            ("content_version", "synthetic-v1"), ("content_version", "g" * 64),
        ]
        for field, value in cases:
            with self.subTest(field=field, value=value):
                rows = copy.deepcopy(self.rows)
                rows[0][field] = value
                with self.assertRaises(ValueError):
                    validate_answer_rows(rows, self.registry)

    def test_unknown_fields_and_empty_input_are_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["name"] = "private"
        for invalid in [rows, [], None, [None]]:
            with self.subTest(invalid=type(invalid).__name__):
                with self.assertRaises(ValueError):
                    validate_answer_rows(invalid, self.registry)

    def test_obsolete_question_index_column_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["question_index"] = 0
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_invalid_or_ambiguous_registry_positions_are_rejected(self):
        key = ("money-basics", self.rows[1]["content_version"], "money-out")
        for position in [None, -1, True, 0]:
            registry = copy.deepcopy(self.registry)
            registry[key]["position"] = position
            with self.subTest(position=position), self.assertRaises(ValueError):
                validate_answer_rows(self.rows, registry)

    def test_postgres_integer_overflow_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["attempt_no"] = 2147483648
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_boolean_seed_is_not_accepted_as_integer(self):
        with self.assertRaises(ValueError):
            generate_synthetic_responses(seed=True)

    def test_duplicate_event_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[1]["event_id"] = rows[0]["event_id"]
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_duplicate_question_within_submission_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[1]["question_id"] = rows[0]["question_id"]
        rows[1]["concept_id"] = rows[0]["concept_id"]
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_unknown_version_question_or_concept_is_rejected(self):
        for field, value in [("content_version", "0" * 64), ("question_id", "unknown"),
                             ("concept_id", "money-out"), ("topic_id", "unknown")]:
            rows = copy.deepcopy(self.rows)
            rows[0][field] = value
            with self.subTest(field=field), self.assertRaises(ValueError):
                validate_answer_rows(rows, self.registry)

    def test_version_is_a_hash_of_the_synthetic_content(self):
        self.assertEqual(self.rows[0]["content_version"], self.metadata["content_version"])
        self.assertEqual(self.metadata["content_version"], synthetic_content_hash())

    def test_a_submission_cannot_mix_versions(self):
        rows = copy.deepcopy(self.rows)
        other_version = "1" * 64
        registry = dict(self.registry)
        registry[("money-basics", other_version, "money-out")] = {"concept_id": "money-out", "position": 1}
        rows[1]["content_version"] = other_version
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, registry)

    def test_pinned_content_orders_equal_timestamps_without_uuid_tiebreak(self):
        rows = list(reversed(self.rows))
        before = copy.deepcopy(rows)
        result = validate_answer_rows(rows, self.registry)
        self.assertEqual(rows, before)
        for left, right in zip(result, result[1:]):
            key = lambda r: (r["learner_id"], r["topic_id"], r["attempt_no"],
                             self.registry[(r["topic_id"], r["content_version"], r["question_id"])]["position"])
            self.assertLess(key(left), key(right))

    def test_timestamp_is_normalized_to_utc(self):
        rows = copy.deepcopy(self.rows)
        for row in rows:
            row["answered_at"] = row["answered_at"].replace("+00:00", "Z")
        self.assertTrue(all(row["answered_at"].endswith("+00:00")
                            for row in validate_answer_rows(rows, self.registry)))

    def test_time_cannot_go_backwards_in_attempt_order(self):
        rows = copy.deepcopy(self.rows)
        rows[2]["answered_at"] = "2025-01-01T00:00:00+00:00"
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_repeated_questions_across_attempts_are_valid(self):
        self.assertEqual(self.rows[0]["question_id"], self.rows[2]["question_id"])
        self.assertNotEqual(self.rows[0]["attempt_no"], self.rows[2]["attempt_no"])
        validate_answer_rows(self.rows, self.registry)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(SyntheticInputTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise AssertionError('SYNTHETIC input validation tests failed')
print('SYNTHETIC input checks passed. No training performed.')


In [ ]:
rows, registry, dataset_metadata = generate_synthetic_responses(seed=42)
validated_rows = validate_answer_rows(rows, registry)
print(LABEL)
print(json.dumps(dataset_metadata, indent=2, ensure_ascii=False))
print(f"SYNTHETIC: {len(validated_rows)} answer events, 12 fictional learners")
print(json.dumps(validated_rows[:2], indent=2))


## Next increment — not implemented

Later: learner-separated fitting/evaluation and draft JSON parameters. Initial app mastery priors are none=0.1, some=0.3, regular=0.5; weak <0.6, practising >=0.6 and <0.95, mastered >=0.95. These rules are recorded only, not applied in this notebook. Fitted synthetic parameters must not automatically replace app priors or influence production recommendations.

Do not describe any synthetic result as real-student accuracy. No export or training in Increment 1.
